# ShapeBench-CUDA Colab GPU Evaluation

This notebook runs ShapeBench-CUDA experiments on a Google Colab GPU runtime.

Use it when you want a lower-friction GPU path than Vast.ai. The evaluator and artifact format are the same project workflow; only the execution backend changes.

Before running:

1. In Colab, choose **Runtime -> Change runtime type -> GPU**.
2. If the GitHub repo is private, add a Colab secret named `GITHUB_TOKEN` with read access to the repo.
3. Run cells from top to bottom.


In [ ]:
# Configuration
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = "https://github.com/nimeshk03/shape_bench_CUDA.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/shape_bench_CUDA")

# Change this to run a different batch.
EXPERIMENT_CONFIG = "configs/phase1_task013_016.json"

# Keep these small for a first Colab smoke run. Increase later for more stable timing.
BENCHMARK_WARMUP = 10
BENCHMARK_ITERS = 50

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_SMOKE_TESTS = True
ALLOW_CPU = False

RUN_ID

In [ ]:
# Helpers
import os
import subprocess
from typing import Sequence
from urllib.parse import quote

def run(command: Sequence[str], *, cwd: Path | None = None, display: Sequence[str] | None = None) -> subprocess.CompletedProcess:
    shown = " ".join(display or command)
    print(f"$ {shown}", flush=True)
    return subprocess.run(command, cwd=cwd, check=True, text=True)

def colab_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value or ""
    except Exception:
        return os.environ.get(name, "")

def repo_clone_url() -> str:
    token = colab_secret("GITHUB_TOKEN")
    if token and REPO_URL.startswith("https://github.com/"):
        safe_token = quote(token, safe="")
        return REPO_URL.replace("https://", f"https://x-access-token:{safe_token}@", 1)
    return REPO_URL


In [ ]:
# Private repo access check. Run this before cloning.
token = colab_secret("GITHUB_TOKEN")
print("GITHUB_TOKEN available:", bool(token))
print("GITHUB_TOKEN length:", len(token or ""))
if not token:
    raise RuntimeError(
        "Missing Colab secret GITHUB_TOKEN. Add it in the Secrets panel, enable notebook access, then rerun the helper cell."
    )

run(["git", "ls-remote", "--heads", repo_clone_url(), BRANCH], display=["git", "ls-remote", "--heads", "<repo-url>", BRANCH])


In [ ]:
# Clone or update the repository.
clone_url = repo_clone_url()
if PROJECT_DIR.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run(
        ["git", "clone", "--branch", BRANCH, clone_url, str(PROJECT_DIR)],
        display=["git", "clone", "--branch", BRANCH, "<repo-url>", str(PROJECT_DIR)],
    )

run(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT_DIR)


In [ ]:
# Install project dependencies. Colab usually provides PyTorch; requirements.txt does not reinstall torch.
run(["python", "-m", "pip", "install", "-U", "pip"], cwd=PROJECT_DIR)
run(["python", "-m", "pip", "install", "-r", "requirements.txt"], cwd=PROJECT_DIR)


In [ ]:
# GPU/CUDA preflight. If this fails, switch the Colab runtime to GPU and reconnect.
run(["nvidia-smi"], cwd=PROJECT_DIR)
run(["nvcc", "--version"], cwd=PROJECT_DIR)
run([
    "python",
    "-c",
    "import torch; print('PyTorch:', torch.__version__); print('CUDA available:', torch.cuda.is_available()); print('CUDA:', torch.version.cuda); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)",
], cwd=PROJECT_DIR)


In [ ]:
# Optional CPU-compatible smoke tests before spending GPU time.
if RUN_SMOKE_TESTS:
    run(["python", "-m", "pytest", "tests/test_experiment_config.py", "tests/test_gpu_eval_batch.py", "tests/test_experiment_analysis.py"], cwd=PROJECT_DIR)


In [ ]:
# Run the selected experiment on the Colab GPU and export compact artifacts.
cmd = [
    "python",
    "scripts/run_colab_eval.py",
    "--experiment", EXPERIMENT_CONFIG,
    "--run-id", RUN_ID,
    "--benchmark-warmup", str(BENCHMARK_WARMUP),
    "--benchmark-iters", str(BENCHMARK_ITERS),
    "--export-artifact",
]
if ALLOW_CPU:
    cmd.append("--allow-cpu")
run(cmd, cwd=PROJECT_DIR)


In [ ]:
# Regenerate the analysis report including the new Colab artifact.
run(["python", "scripts/generate_initial_findings.py"], cwd=PROJECT_DIR)

summary_path = PROJECT_DIR / "results" / "experiments" / RUN_ID / "summary.json"
raw_path = PROJECT_DIR / "results" / "experiments" / RUN_ID / "raw_results.jsonl"
print("summary:", summary_path)
print("raw:", raw_path)
print("report:", PROJECT_DIR / "report" / "initial_findings.md")


In [ ]:
# Quick result preview.
import json

summary = json.loads((PROJECT_DIR / "results" / "experiments" / RUN_ID / "summary.json").read_text())
metadata = summary.get("run_metadata", summary.get("vast_run_metadata", {}))
print("backend:", metadata.get("backend"))
print("source commit:", summary["source_commit"][:12])
print("records:", summary["raw_results"]["record_count"])
print("aggregates:")
for row in summary["aggregates"]:
    print(row["task_id"], row["prompt_mode"], f"{row['passed']}/{row['checks']}", "mean_speedup=", row["mean_speedup_vs_eager"])


In [ ]:
# Package the Colab run, exported artifact, and report for download.
import zipfile

zip_path = Path("/content") / f"shapebench_colab_{RUN_ID}.zip"
paths_to_include = [
    PROJECT_DIR / "results" / "colab_runs" / RUN_ID,
    PROJECT_DIR / "results" / "experiments" / RUN_ID,
    PROJECT_DIR / "report" / "initial_findings.md",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in paths_to_include:
        if path.is_dir():
            for file_path in path.rglob("*"):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(PROJECT_DIR.parent))
        elif path.is_file():
            archive.write(path, path.relative_to(PROJECT_DIR.parent))

print("Created:", zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download manually from:", zip_path)
